# 03 — Forecasting Layer 1: Base Forecast

Division-level daily sales forecasting using `division_daily` as the primary table.

Three models compared on RMSE and MAPE:
1. **ARIMA** (statsmodels) — baseline
2. **Prophet** — seasonality + event regressors
3. **XGBoost** — multi-factor with lag features

SHAP values on XGBoost power the dashboard driver attribution waterfall chart.
Trained artifacts saved to `src/models/`.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import warnings
warnings.filterwarnings('ignore')

from src.utils.db_connect import get_engine
engine = get_engine()
print('Connected')

## 1. Load and Prepare Data

In [ ]:
# Load division_daily joined with daily_sales context features
df = pd.read_sql("""
    SELECT
        dd.date, dd.division_code, dd.division_name, dd.department,
        dd.sales_amt, dd.gp_amt, dd.gp_ratio, dd.num_transactions,
        ds.day_of_week, ds.event_flag, ds.temperature_index,
        ds.is_public_holiday, ds.trading_hours, ds.promo_flag
    FROM division_daily dd
    JOIN daily_sales ds ON dd.date = ds.date
    ORDER BY dd.date, dd.division_code
""", engine, parse_dates=['date'])

print(f'Loaded {len(df):,} rows | {df["division_code"].nunique()} divisions | {df["date"].nunique()} days')
df.head(3)

## 2. Feature Engineering

In [ ]:
from src.features.feature_engineering import build_features

df_feat = build_features(df)
print(df_feat.shape)
df_feat.head(3)

## 3. Train/Test Split (last 8 weeks = test)

In [ ]:
split_date = df_feat['date'].max() - pd.Timedelta(weeks=8)
train = df_feat[df_feat['date'] <= split_date].copy()
test  = df_feat[df_feat['date'] >  split_date].copy()
print(f'Train: {train["date"].min().date()} → {train["date"].max().date()} ({len(train):,} rows)')
print(f'Test:  {test["date"].min().date()} → {test["date"].max().date()} ({len(test):,} rows)')

## 4. ARIMA Baseline (per division)

In [ ]:
from statsmodels.tsa.arima.model import ARIMA

DEMO_DIV = 34  # Men's Cut & Sewn — top revenue driver

div_train = train[train['division_code'] == DEMO_DIV].set_index('date')['sales_amt']
div_test  = test[test['division_code']   == DEMO_DIV].set_index('date')['sales_amt']

arima_model = ARIMA(div_train, order=(7, 1, 2), seasonal_order=(1, 1, 1, 7))
arima_fit = arima_model.fit()

arima_pred = arima_fit.forecast(steps=len(div_test))
arima_pred.index = div_test.index

arima_mape = np.mean(np.abs((div_test - arima_pred) / div_test)) * 100
arima_rmse = np.sqrt(np.mean((div_test - arima_pred)**2))
print(f'ARIMA (Div {DEMO_DIV}) — MAPE: {arima_mape:.2f}% | RMSE: ${arima_rmse:,.0f}')

## 5. Prophet (seasonality + event regressors)

In [ ]:
from prophet import Prophet
import pickle
from pathlib import Path
MODEL_DIR = Path('../src/models')
MODEL_DIR.mkdir(exist_ok=True)

def train_prophet(div_code: int, train_df: pd.DataFrame, test_df: pd.DataFrame):
    d_train = train_df[train_df['division_code'] == div_code][['date','sales_amt','event_flag']].copy()
    d_test  = test_df[test_df['division_code']   == div_code][['date','sales_amt','event_flag']].copy()
    d_train = d_train.rename(columns={'date': 'ds', 'sales_amt': 'y'})
    d_test  = d_test.rename(columns={'date': 'ds', 'sales_amt': 'y'})

    # Encode event flags as binary regressors
    for ev in ['Arigato', 'Christmas', 'EOFY', 'Vivid', 'LongWeekend']:
        d_train[ev] = (d_train['event_flag'] == ev).astype(int)
        d_test[ev]  = (d_test['event_flag']  == ev).astype(int)

    m = Prophet(yearly_seasonality=True, weekly_seasonality=True,
                 daily_seasonality=False, interval_width=0.70)
    for ev in ['Arigato', 'Christmas', 'EOFY', 'Vivid', 'LongWeekend']:
        m.add_regressor(ev)
    m.fit(d_train)

    future = d_test[['ds'] + ['Arigato','Christmas','EOFY','Vivid','LongWeekend']]
    forecast = m.predict(future)
    pred = forecast.set_index('ds')['yhat']
    actual = d_test.set_index('ds')['y']

    mape = np.mean(np.abs((actual - pred) / actual)) * 100
    rmse = np.sqrt(np.mean((actual - pred)**2))

    with open(MODEL_DIR / f'prophet_div{div_code}.pkl', 'wb') as f:
        pickle.dump(m, f)

    return m, forecast, mape, rmse

m_prophet, fc_prophet, prophet_mape, prophet_rmse = train_prophet(DEMO_DIV, train, test)
print(f'Prophet (Div {DEMO_DIV}) — MAPE: {prophet_mape:.2f}% | RMSE: ${prophet_rmse:,.0f}')

## 6. XGBoost Multi-Factor Model

In [ ]:
import xgboost as xgb
import shap
import pickle

FEATURE_COLS = [
    'division_code', 'temperature_index', 'trading_hours',
    'is_public_holiday', 'promo_flag',
    'dow_num',                          # 0=Mon … 6=Sun
    'month', 'day_of_year',
    'event_Arigato', 'event_Christmas', 'event_EOFY', 'event_Vivid', 'event_LongWeekend',
    'lag_7', 'lag_14', 'lag_28',
    'rolling_7_mean', 'rolling_14_mean',
]

X_train = train[FEATURE_COLS].fillna(0)
y_train = train['sales_amt']
X_test  = test[FEATURE_COLS].fillna(0)
y_test  = test['sales_amt']

xgb_model = xgb.XGBRegressor(
    n_estimators=500, learning_rate=0.05, max_depth=6,
    subsample=0.8, colsample_bytree=0.8,
    objective='reg:squarederror', random_state=42,
    early_stopping_rounds=20, eval_metric='rmse'
)

split_val = int(len(X_train) * 0.85)
xgb_model.fit(
    X_train[:split_val], y_train[:split_val],
    eval_set=[(X_train[split_val:], y_train[split_val:])],
    verbose=50
)

y_pred_xgb = xgb_model.predict(X_test)
xgb_mape = np.mean(np.abs((y_test - y_pred_xgb) / y_test)) * 100
xgb_rmse = np.sqrt(np.mean((y_test - y_pred_xgb)**2))
print(f'XGBoost (all divs) — MAPE: {xgb_mape:.2f}% | RMSE: ${xgb_rmse:,.0f}')

with open(MODEL_DIR / 'xgb_layer1.pkl', 'wb') as f:
    pickle.dump(xgb_model, f)

## 7. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['ARIMA', 'Prophet', 'XGBoost'],
    'MAPE (%)': [arima_mape, prophet_mape, xgb_mape],
    'RMSE (AUD)': [arima_rmse, prophet_rmse, xgb_rmse],
})
print(comparison.round(2))

fig = make_subplots(rows=1, cols=2, subplot_titles=['MAPE (%)', 'RMSE (AUD)'])
colors = ['#FF9800', '#2196F3', '#4CAF50']
fig.add_trace(go.Bar(x=comparison['Model'], y=comparison['MAPE (%)'],
                      marker_color=colors, name='MAPE'), row=1, col=1)
fig.add_trace(go.Bar(x=comparison['Model'], y=comparison['RMSE (AUD)'],
                      marker_color=colors, name='RMSE', showlegend=False), row=1, col=2)
fig.update_yaxes(ticksuffix='%', row=1, col=1)
fig.update_yaxes(tickformat='$,.0f', row=1, col=2)
fig.update_layout(title='Model Comparison — MAPE and RMSE on Test Set', showlegend=False)
fig.show()

## 8. SHAP Driver Attribution

In [ ]:
explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_test)

# Global feature importance
shap_df = pd.DataFrame(shap_values, columns=FEATURE_COLS)
mean_abs_shap = shap_df.abs().mean().sort_values(ascending=False)

fig = px.bar(mean_abs_shap, orientation='h',
             title='XGBoost Feature Importance (Mean |SHAP|)',
             labels={'value': 'Mean |SHAP|', 'index': 'Feature'})
fig.update_layout(height=500)
fig.show()

In [ ]:
# Single-day SHAP waterfall — example forecast explanation
sample_idx = 0
sample_shap = shap_values[sample_idx]
top_n = 8
top_idx = np.argsort(np.abs(sample_shap))[-top_n:][::-1]
top_features = [FEATURE_COLS[i] for i in top_idx]
top_values = [sample_shap[i] for i in top_idx]

fig = go.Figure(go.Waterfall(
    name='SHAP', orientation='v',
    measure=['relative'] * len(top_features) + ['total'],
    x=top_features + ['Total'],
    y=top_values + [sum(sample_shap)],
    connector={'line': {'color': 'rgb(63,63,63)'}},
))
fig.update_layout(
    title=f'SHAP Waterfall — Sample Forecast Explanation (Div {DEMO_DIV})',
    yaxis_tickformat='$,.0f'
)
fig.show()

# Save SHAP explainer for dashboard
import pickle
with open(MODEL_DIR / 'shap_explainer.pkl', 'wb') as f:
    pickle.dump(explainer, f)
np.save(MODEL_DIR / 'feature_cols.npy', np.array(FEATURE_COLS))
print('SHAP explainer saved.')

## 9. Forecast Plot: Actual vs XGBoost (all divisions)

In [ ]:
test_plot = test.copy()
test_plot['predicted'] = y_pred_xgb
test_plot['error_pct'] = (test_plot['sales_amt'] - test_plot['predicted']) / test_plot['sales_amt'] * 100

# Aggregate to daily store total for a clean overview chart
daily_agg = test_plot.groupby('date')[['sales_amt', 'predicted']].sum().reset_index()

fig = go.Figure()
fig.add_trace(go.Scatter(x=daily_agg['date'], y=daily_agg['sales_amt'],
                          mode='lines', name='Actual', line=dict(color='#1f77b4')))
fig.add_trace(go.Scatter(x=daily_agg['date'], y=daily_agg['predicted'],
                          mode='lines', name='XGBoost Forecast',
                          line=dict(color='#2ca02c', dash='dash')))
fig.update_layout(title='XGBoost: Actual vs Forecast — Test Period (All Divisions)',
                  yaxis_tickformat='$,.0f', hovermode='x unified')
fig.show()